# GEE_XEE_sen2_chl.ipynb

**Sentinel-2 Water Chlorophyll_a Index using Python API (Xee)**

> Tutorial Code Adapted by Souleymane Mamana Nouri Souley

This notebook demonstrates how to compute and visualize a chlorophyll-*a* proxy over water bodies using Sentinel-2 Level-2A data in Google Earth Engine, bridged to xarray via Xee.


In [ ]:
import ee
import geemap
import xarray as xr
import matplotlib.pyplot as plt

# Install xee if missing (optional, uncomment in hosted environments)
# !pip install xee
import xee

## Authenticate & Initialize Earth Engine
Replace the `project` value with your GEE Cloud Project ID if needed. The High Volume endpoint is used here.

In [ ]:
ee.Authenticate()
ee.Initialize(
    project="platinum-pager-426715-c6",
    opt_url="https://earthengine-highvolume.googleapis.com",
)

## Draw ROI
Use the map toolbar to draw a polygon. The last feature is read as the ROI.

In [ ]:
map = geemap.Map(basemap="TERRAIN")
map

# After drawing on the map, capture ROI
roi = map.draw_last_feature.geometry()
roi

## Define processing function (cloud & water masks + Chl-a proxy)

In [ ]:
def sen2q(img):
    cloud = img.select("probability")
    clear_pixels = cloud.lt(20)
    bands = img.select("B.*").multiply(0.0001)
    ndwi = bands.normalizedDifference(["B3", "B8"]).rename("ndwi")
    water_body = ndwi.gt(0.1)
    chl = bands.expression(
        "4.26 * ((b3 / b1) ** 3.94)",
        {"b1": bands.select("B1"), "b3": bands.select("B3")},
    ).rename("chl")
    return (
        chl.updateMask(clear_pixels)
        .updateMask(water_body)
        .copyProperties(img, ["system:time_start"])
    )

## Build ImageCollection (2024) and apply function

In [ ]:
sen2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .linkCollection(
        ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY"), "probability"
    )
    .filterDate("2024", "2025")
    .filterBounds(roi)
    .map(sen2q)
)
sen2

## Convert to xarray via Xee and sort by time

In [ ]:
ds = xr.open_dataset(sen2, engine="ee", crs="EPSG:4326", scale=0.001, geometry=roi)
ds = ds.sortby("time")
ds

## Monthly resample (median) and plot facets

In [ ]:
ds_monthly = ds.resample(time="M").median("time")

fig = ds_monthly.chl.plot(
    x="lon",
    y="lat",
    col="time",
    col_wrap=6,
    robust=True,
    vmin=2,
    vmax=30,
    cmap="rainbow",
    levels=20,
)
plt.savefig("sen2_chl.png", dpi=360, bbox_inches="tight")
"Saved sen2_chl.png"

## Notes
- The Chl-a expression is a proxy for demonstration; calibrate with in-situ data for quantitative use.
- Adjust cloud threshold, NDWI threshold, and scale to suit your site.
